# 🚀 Nexus Audio - Treino RÁPIDO

**Usa tokens pré-processados** = ~10x mais rápido!

⚡ Cada step agora leva segundos, não minutos.

In [ ]:
!nvidia-smi

# Mamba é opcional (treino funciona sem ele, só mais lento)
!pip install -q torch torchaudio einops encodec accelerate
!pip install -q causal-conv1d mamba-ssm || echo 'Mamba fallback - OK'

In [ ]:
import torch
import torch.nn as nn
import os
import sys
from pathlib import Path
from tqdm.notebook import tqdm
import time

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPUs: {torch.cuda.device_count()}")

In [ ]:
# Carregar código
sys.path.insert(0, "/kaggle/input/nexus-audio-code")
from src.model import SiMBATherapeutic
print("✅ Código carregado!")

In [ ]:
# ⚡ CAMINHO DOS TOKENS PRÉ-PROCESSADOS
TOKENS_PATH = "/kaggle/input/nexus-tokens"  # <-- TROCAR pelo nome do seu dataset!

token_files = list(Path(TOKENS_PATH).glob("**/*.pt"))
print(f"✅ Tokens encontrados: {len(token_files)}")

# Ver shape de um token
if token_files:
    sample = torch.load(token_files[0])
    print(f"Shape: {sample.shape}")

In [ ]:
# Configuração
CONFIG = {
    "model": {
        "d_model": 384,
        "n_layers": 6,
        "d_state": 16,
        "d_conv": 4,
        "expand": 2,
        "max_seq_len": 4096,
    },
    "audio": {
        "sample_rate": 24000,
        "encodec_bandwidth": 6.0,
    },
    "therapeutic": {"use_biofeedback": False},
}

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# Criar modelo
model = SiMBATherapeutic.from_config(CONFIG)

if torch.cuda.device_count() > 1:
    print(f"🚀 DataParallel com {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
    
model = model.cuda()
print(f"Parâmetros: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Dataset de tokens (SUPER RÁPIDO!)
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

class TokenDataset(Dataset):
    """Dataset que carrega tokens pré-processados (RÁPIDO!)"""
    def __init__(self, token_dir):
        self.files = list(Path(token_dir).glob("**/*.pt"))
        print(f"Dataset: {len(self.files)} tokens")
        
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        tokens = torch.load(self.files[idx])
        # Flatten se necessário (codebooks x seq -> seq)
        if tokens.dim() == 2:
            # Usar só o primeiro codebook ou flatten
            tokens = tokens[0]  # [seq_len]
        return tokens.long()

In [ ]:
def train_fast(model, token_dir, phase_name, max_steps, lr, batch_size=8, accum=4):
    print(f"\n{'='*50}")
    print(f"🚀 {phase_name}")
    print(f"{'='*50}")
    
    dataset = TokenDataset(token_dir)
    loader = DataLoader(
        dataset, 
        batch_size=batch_size,
        shuffle=True, 
        num_workers=2,  # Pode usar workers agora! Tokens são CPU
        pin_memory=True,
    )
    
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scheduler = CosineAnnealingLR(optimizer, T_max=max_steps, eta_min=lr*0.1)
    scaler = GradScaler('cuda')
    
    model.train()
    global_step = 0
    running_loss = 0
    start = time.time()
    
    # Checkpoint inicial
    state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(state, f"{CHECKPOINT_DIR}/{phase_name}_step0.pt")
    print("💾 Checkpoint inicial salvo!")
    
    pbar = tqdm(total=max_steps, desc=phase_name)
    
    while global_step < max_steps:
        for batch_idx, tokens in enumerate(loader):
            tokens = tokens.cuda()
            
            with autocast('cuda'):
                outputs = model(tokens=tokens, labels=tokens)
                loss = outputs["loss"]
                if isinstance(loss, tuple):
                    loss = loss[0]
                loss = loss.mean() / accum
            
            scaler.scale(loss).backward()
            running_loss += loss.item()
            
            if (batch_idx + 1) % accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                
                global_step += 1
                pbar.update(1)
                
                if global_step % 50 == 0:
                    avg_loss = running_loss / 50
                    pbar.set_postfix({"loss": f"{avg_loss:.4f}"})
                    running_loss = 0
                
                if global_step % 500 == 0:
                    state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
                    torch.save(state, f"{CHECKPOINT_DIR}/{phase_name}_step{global_step}.pt")
                    elapsed = (time.time() - start) / 60
                    print(f"\n💾 Step {global_step} | Tempo: {elapsed:.1f}min")
                
                if global_step >= max_steps:
                    break
    
    pbar.close()
    elapsed = (time.time() - start) / 60
    print(f"\n✅ {phase_name} completa! Tempo: {elapsed:.1f} min")
    
    state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(state, f"{CHECKPOINT_DIR}/{phase_name}_final.pt")
    return model

In [ ]:
# 🚀 TREINAR!
model = train_fast(
    model=model,
    token_dir=TOKENS_PATH,
    phase_name="pretrain_fma",
    max_steps=5000,
    lr=3e-4,
)

In [ ]:
# Salvar modelo final
state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
torch.save({"model_state_dict": state, "config": CONFIG}, f"{CHECKPOINT_DIR}/nexus_trained.pt")

!cp /kaggle/working/checkpoints/*.pt /kaggle/working/
!ls -la /kaggle/working/*.pt
print("\n✅ Modelo salvo!")

---
# 🎵 Testar Geração

In [ ]:
import torchaudio

gen_model = model.module if isinstance(model, nn.DataParallel) else model
gen_model.eval()

print("Gerando 10s de música...")
with torch.no_grad():
    waveform = gen_model.generate(duration_seconds=10.0, temperature=0.9)

if waveform.dim() == 3:
    waveform = waveform.squeeze(0)
torchaudio.save("/kaggle/working/output.wav", waveform.cpu(), 24000)
print("✅ Áudio salvo!")

In [ ]:
from IPython.display import Audio
Audio("/kaggle/working/output.wav")